<a href="https://colab.research.google.com/github/dominiksakic/NETworkingMay/blob/main/23_nlp_sequence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80.2M  100 80.2M    0     0  31.4M      0  0:00:02  0:00:02 --:--:-- 31.4M


In [2]:
!rm -r aclImdb/train/unsup

In [3]:
import os, pathlib, shutil, random
from tensorflow import keras

# Extract data
base_dir = pathlib.Path("aclImdb")
val_dir = base_dir / "val"
train_dir = base_dir / "train"

for category in ("neg", "pos"):
  os.makedirs(val_dir / category)
  files = os.listdir(train_dir / category)
  random.Random(1337).shuffle(files)
  num_val_samples = int(0.2 * len(files))
  val_files = files[-num_val_samples:]
  for fname in val_files:
    shutil.move(train_dir / category / fname,
                val_dir / category / fname)

# Create sets
batch_size = 32

train_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/train", batch_size=batch_size)
val_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/val", batch_size=batch_size)
test_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/test", batch_size=batch_size)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


In [4]:
from tensorflow.keras import layers

max_length = 600
max_tokens = 20000
text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length = max_length,
)

text_only_train_ds = train_ds.map(lambda x, y: x)
text_vectorization.adapt(text_only_train_ds)

int_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
int_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
int_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

In [5]:
import tensorflow as tf
from tensorflow.keras import layers

# Custom Layer for accepting the symbolic tensor inputs
class OneHotLayer(layers.Layer):
  def __init__(self, depth, **kwargs):
    super().__init__(**kwargs)
    self.depth = depth

  def call(self, inputs):
    return tf.one_hot(inputs, depth=self.depth)


inputs = keras.Input(shape=(None,), dtype="int64")
embedded = OneHotLayer(depth=max_tokens)(inputs)
x = layers.Bidirectional(layers.LSTM(32))(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)

model.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=["acc"])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ one_hot_layer (OneHotLayer)     │ (None, None, 20000)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 64)             │     5,128,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,128,513 (19.56 MB)

 Trainable params: 5,128,513 (19.56 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
callbacks = [
  keras.callbacks.ModelCheckpoint("one_hot_bidir_lstm.keras",
  save_best_only=True)
]

model.fit(int_train_ds,
          validation_data=int_val_ds,
          epochs=10,callbacks=callbacks)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 219s 344ms/step - acc: 0.5882 - loss: 0.6535 - val_acc: 0.8184 - val_loss: 0.4174
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 279s 374ms/step - acc: 0.8453 - loss: 0.4008 - val_acc: 0.8266 - val_loss: 0.5416
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 262s 374ms/step - acc: 0.8860 - loss: 0.3255 - val_acc: 0.8722 - val_loss: 0.3639
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 215s 344ms/step - acc: 0.9109 - loss: 0.2569 - val_acc: 0.8822 - val_loss: 0.3292
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 280s 373ms/step - acc: 0.9237 - loss: 0.2233 - val_acc: 0.8872 - val_loss: 0.2937
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 233s 372ms/step - acc: 0.9317 - loss: 0.1974 - val_acc: 0.8746 - val_loss: 0.3168
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 262s 373ms/step - acc: 0.9429 - loss: 0.1732 - val_acc: 0.8776 - val_loss: 0.3531
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 262s 373ms/step - acc: 0.9528 - loss: 0.1511 - val_acc: 0.8810 - val_loss: 0.3696
Epoch 9/10
625/625 ━━━━━

In [8]:
model = keras.models.load_model("one_hot_bidir_lstm.keras",
                                custom_objects={"OneHotLayer": OneHotLayer})
print(f"Test acc: {model.evaluate(int_test_ds)[1]:.3f}")

782/782 ━━━━━━━━━━━━━━━━━━━━ 116s 147ms/step - acc: 0.8740 - loss: 0.3121
Test acc: 0.875


- Using hot encoding for sequnce models is very slow!

**Simplified Example**
- "The cat sat!" # Original Sentence
- [0, 1, 2] # Integer encoded
- [[1 0 0], [0 1 0], [0 0 0]] # one hot encoded

**For above code**
- max_length = 600
- dictonary = 20000
- results = 600 * 20000 = 12.000.000.000 floats  